# 09 RL Environment Test

## 목표
직접 만든 Snake 환경에서 학습 전에 heuristic agent를 먼저 돌려 보며, 상태(state), 행동(action), 보상(reward), 종료(done) 규칙이 실제로 원하는 대로 작동하는지 확인한다.
특히 음식 방향으로 직관적으로 움직이는 정책을 넣었을 때도 벽 충돌, 몸통 충돌, timeout 규칙이 로그에 깔끔하게 드러나는지를 검증한다.


## 1. SnakeGrid 규칙
- Grid 크기: `6 x 6`
- Snake 시작 길이: `2`
- 행동(action): `0=상`, `1=하`, `2=좌`, `3=우`
- Food: `1개`
- 보상(reward): Food 획득 `+10`, 충돌 `-10`, 일반 이동 `-0.1`, timeout `-2`
- 종료(done): 벽 충돌, 몸통 충돌, 혹은 episode가 너무 길어졌을 때 timeout
- timeout 기준: `max_steps = grid_size * grid_size * 2`

이번 노트북의 목적은 학습 알고리즘을 붙이기 전에, 환경이 원하는 대로 반응하는지 heuristic agent 로그로 먼저 확인하는 것이다.


In [1]:
import random

import pandas as pd
from IPython.display import display

SEED = 7
GRID_SIZE = 6
MAX_STEPS_MULTIPLIER = 2

ACTION_NAMES = {0: "UP", 1: "DOWN", 2: "LEFT", 3: "RIGHT"}
ACTION_TO_DELTA = {
    0: (-1, 0),
    1: (1, 0),
    2: (0, -1),
    3: (0, 1),
}
OPPOSITE_ACTION = {0: 1, 1: 0, 2: 3, 3: 2}

REWARD_CONFIG = {
    "food": 10.0,
    "collision": -10.0,
    "step": -0.1,
    "timeout": -2.0,
}


def format_state(state):
    hx, hy, fx, fy, direction, length = state
    return (
        f"(hx={hx}, hy={hy}, fx={fx}, fy={fy}, "
        f"dir={ACTION_NAMES[direction]}, len={length})"
    )


In [2]:
class SnakeGridEnv:
    def __init__(self, size=6, reward_config=None, max_steps=None, seed=None):
        self.size = size
        self.reward_config = dict(reward_config or REWARD_CONFIG)
        self.max_steps = max_steps or size * size * MAX_STEPS_MULTIPLIER
        self.rng = random.Random(seed)
        self.snake = []
        self.food_pos = None
        self.direction = 3
        self.steps = 0
        self.foods_eaten = 0

    def reset(self, seed=None):
        if seed is not None:
            self.rng.seed(seed)

        center_x = self.size // 2
        center_y = self.size // 2
        self.snake = [(center_x, center_y), (center_x, center_y - 1)]
        self.direction = 3
        self.steps = 0
        self.foods_eaten = 0
        self.food_pos = self._spawn_food()
        return self._get_state()

    def step(self, action):
        action = self._normalize_action(action)
        next_head = self._next_head(action)
        self.steps += 1

        if not self._in_bounds(next_head):
            reward = self.reward_config["collision"]
            info = {
                "event": "wall_collision",
                "length": len(self.snake),
                "foods_eaten": self.foods_eaten,
                "max_steps": self.max_steps,
            }
            return self._get_state(), reward, True, info

        grow = next_head == self.food_pos
        body_to_check = set(self.snake if grow else self.snake[:-1])
        if next_head in body_to_check:
            reward = self.reward_config["collision"]
            info = {
                "event": "self_collision",
                "length": len(self.snake),
                "foods_eaten": self.foods_eaten,
                "max_steps": self.max_steps,
            }
            return self._get_state(), reward, True, info

        self.snake.insert(0, next_head)
        self.direction = action
        done = False

        if grow:
            self.foods_eaten += 1
            reward = self.reward_config["food"]
            if len(self.snake) == self.size * self.size:
                info = {
                    "event": "board_cleared",
                    "length": len(self.snake),
                    "foods_eaten": self.foods_eaten,
                    "max_steps": self.max_steps,
                }
                return self._get_state(), reward, True, info
            self.food_pos = self._spawn_food()
            event = "food"
        else:
            self.snake.pop()
            reward = self.reward_config["step"]
            event = "move"

        if self.steps >= self.max_steps:
            reward = self.reward_config["timeout"]
            done = True
            event = "timeout"

        info = {
            "event": event,
            "length": len(self.snake),
            "foods_eaten": self.foods_eaten,
            "max_steps": self.max_steps,
        }
        return self._get_state(), reward, done, info

    def _get_state(self):
        hx, hy = self.snake[0]
        fx, fy = self.food_pos
        return (hx, hy, fx, fy, self.direction, len(self.snake))

    def _spawn_food(self):
        candidates = [
            (x, y)
            for x in range(self.size)
            for y in range(self.size)
            if (x, y) not in self.snake
        ]
        return self.rng.choice(candidates)

    def _normalize_action(self, action):
        if action not in ACTION_TO_DELTA:
            raise ValueError(f"invalid action: {action}")
        if len(self.snake) > 1 and action == OPPOSITE_ACTION[self.direction]:
            return self.direction
        return action

    def _next_head(self, action):
        dx, dy = ACTION_TO_DELTA[action]
        head_x, head_y = self.snake[0]
        return (head_x + dx, head_y + dy)

    def _in_bounds(self, pos):
        return 0 <= pos[0] < self.size and 0 <= pos[1] < self.size

    def render(self):
        grid = [[" . " for _ in range(self.size)] for _ in range(self.size)]
        for idx, (x, y) in enumerate(self.snake):
            grid[x][y] = " H " if idx == 0 else " S "
        fx, fy = self.food_pos
        grid[fx][fy] = " F "
        print(
            f"step={self.steps}, foods={self.foods_eaten}, "
            f"state={format_state(self._get_state())}, max_steps={self.max_steps}"
        )
        for row in grid:
            print("".join(row))


In [3]:
env = SnakeGridEnv(size=GRID_SIZE, reward_config=REWARD_CONFIG, seed=SEED)
initial_state = env.reset(seed=SEED)
print("Initial state:", format_state(initial_state))
print("Max steps per episode:", env.max_steps)
env.render()


Initial state: (hx=3, hy=3, fx=3, fy=4, dir=RIGHT, len=2)
Max steps per episode: 72
step=0, foods=0, state=(hx=3, hy=3, fx=3, fy=4, dir=RIGHT, len=2), max_steps=72
 .  .  .  .  .  . 
 .  .  .  .  .  . 
 .  .  .  .  .  . 
 .  .  S  H  F  . 
 .  .  .  .  .  . 
 .  .  .  .  .  . 


## 2. Heuristic Agent 아이디어
학습 전 검증 단계에서는 무작정 random policy를 돌리는 것보다, 사람이 봤을 때 그럴듯한 정책을 하나 먼저 넣어 보는 것이 도움이 된다.

여기서는 다음 규칙을 쓰는 heuristic agent를 사용한다.
- 바로 반대 방향으로 꺾어 자기 몸통으로 되돌아가는 행동은 피한다.
- 한 step 뒤 벽이나 몸통에 부딪히는 행동은 우선 제외한다.
- 안전한 후보 중에서 food와의 Manhattan distance를 가장 많이 줄이는 행동을 선택한다.
- 동점이면 현재 진행 방향을 유지하는 쪽을 우선한다.

이 정책은 완벽하지 않지만, 환경의 보상/종료 규칙이 잘 정의되어 있는지 빠르게 확인하기에 좋다.


In [4]:
def manhattan(pos_a, pos_b):
    return abs(pos_a[0] - pos_b[0]) + abs(pos_a[1] - pos_b[1])


def is_safe_action(env, action):
    normalized_action = env._normalize_action(action)
    next_head = env._next_head(normalized_action)
    if not env._in_bounds(next_head):
        return False

    grow = next_head == env.food_pos
    body_to_check = set(env.snake if grow else env.snake[:-1])
    return next_head not in body_to_check


def select_heuristic_action(env):
    candidates = []

    for action in ACTION_NAMES:
        normalized_action = env._normalize_action(action)
        next_head = env._next_head(normalized_action)
        safe = is_safe_action(env, action)
        distance = manhattan(next_head, env.food_pos)
        same_direction = int(normalized_action == env.direction)
        candidates.append(
            {
                "action": normalized_action,
                "safe": safe,
                "distance": distance,
                "same_direction": same_direction,
            }
        )

    safe_candidates = [item for item in candidates if item["safe"]]
    pool = safe_candidates if safe_candidates else candidates
    best = min(pool, key=lambda item: (item["distance"], -item["same_direction"], item["action"]))
    return best["action"]


In [5]:
def run_heuristic_policy(env, episodes=3, seed=SEED, verbose=True):
    step_logs = []
    episode_rows = []

    for episode in range(1, episodes + 1):
        state = env.reset(seed=seed + episode)
        episode_reward = 0.0
        max_length = len(env.snake)

        if verbose:
            print(f"\n[heuristic] episode={episode}, start_state={format_state(state)}")

        for step in range(1, env.max_steps + 1):
            action = select_heuristic_action(env)
            next_state, reward, done, info = env.step(action)
            episode_reward += reward
            max_length = max(max_length, info["length"])

            row = {
                "episode": episode,
                "step": step,
                "state": state,
                "action": ACTION_NAMES[action],
                "next_state": next_state,
                "reward": round(reward, 2),
                "done": done,
                "event": info["event"],
                "length": info["length"],
                "foods_eaten": info["foods_eaten"],
            }
            step_logs.append(row)

            if verbose:
                print(
                    f"  step={step:02d}, state={format_state(state)}, "
                    f"action={ACTION_NAMES[action]:>5s}, next_state={format_state(next_state)}, "
                    f"reward={reward:5.1f}, done={done}, event={info['event']}"
                )

            state = next_state
            if done:
                break

        episode_rows.append(
            {
                "episode": episode,
                "episode_reward": round(episode_reward, 2),
                "steps": step,
                "foods_eaten": info["foods_eaten"],
                "max_length": max_length,
                "termination": info["event"],
                "max_steps": env.max_steps,
            }
        )

        if verbose:
            print(
                f"  summary -> steps={step:02d}, total_reward={episode_reward:6.2f}, "
                f"foods={info['foods_eaten']}, max_length={max_length}, termination={info['event']}"
            )

    return pd.DataFrame(step_logs), pd.DataFrame(episode_rows)


In [6]:
test_env = SnakeGridEnv(size=GRID_SIZE, reward_config=REWARD_CONFIG, seed=SEED)
heuristic_logs_df, heuristic_summary_df = run_heuristic_policy(test_env, episodes=3, seed=SEED, verbose=True)

display(heuristic_logs_df.head(40))
display(heuristic_logs_df[heuristic_logs_df["done"]])
display(heuristic_summary_df)



[heuristic] episode=1, start_state=(hx=3, hy=3, fx=2, fy=2, dir=RIGHT, len=2)
  step=01, state=(hx=3, hy=3, fx=2, fy=2, dir=RIGHT, len=2), action=   UP, next_state=(hx=2, hy=3, fx=2, fy=2, dir=UP, len=2), reward= -0.1, done=False, event=move
  step=02, state=(hx=2, hy=3, fx=2, fy=2, dir=UP, len=2), action= LEFT, next_state=(hx=2, hy=2, fx=4, fy=2, dir=LEFT, len=3), reward= 10.0, done=False, event=food
  step=03, state=(hx=2, hy=2, fx=4, fy=2, dir=LEFT, len=3), action= DOWN, next_state=(hx=3, hy=2, fx=4, fy=2, dir=DOWN, len=3), reward= -0.1, done=False, event=move
  step=04, state=(hx=3, hy=2, fx=4, fy=2, dir=DOWN, len=3), action= DOWN, next_state=(hx=4, hy=2, fx=4, fy=4, dir=DOWN, len=4), reward= 10.0, done=False, event=food
  step=05, state=(hx=4, hy=2, fx=4, fy=4, dir=DOWN, len=4), action=RIGHT, next_state=(hx=4, hy=3, fx=4, fy=4, dir=RIGHT, len=4), reward= -0.1, done=False, event=move
  step=06, state=(hx=4, hy=3, fx=4, fy=4, dir=RIGHT, len=4), action=RIGHT, next_state=(hx=4, hy=4,

,episode,step,state,action,next_state,reward,done,event,length,foods_eaten
0,1,1,"(3, 3, 2, 2, 3, 2)",UP,"(2, 3, 2, 2, 0, 2)",-0.1,False,move,2,0
1,1,2,"(2, 3, 2, 2, 0, 2)",LEFT,"(2, 2, 4, 2, 2, 3)",10.0,False,food,3,1
2,1,3,"(2, 2, 4, 2, 2, 3)",DOWN,"(3, 2, 4, 2, 1, 3)",-0.1,False,move,3,1
3,1,4,"(3, 2, 4, 2, 1, 3)",DOWN,"(4, 2, 4, 4, 1, 4)",10.0,False,food,4,2
4,1,5,"(4, 2, 4, 4, 1, 4)",RIGHT,"(4, 3, 4, 4, 3, 4)",-0.1,False,move,4,2
5,1,6,"(4, 3, 4, 4, 3, 4)",RIGHT,"(4, 4, 0, 4, 3, 5)",10.0,False,food,5,3
6,1,7,"(4, 4, 0, 4, 3, 5)",UP,"(3, 4, 0, 4, 0, 5)",-0.1,False,move,5,3
7,1,8,"(3, 4, 0, 4, 0, 5)",UP,"(2, 4, 0, 4, 0, 5)",-0.1,False,move,5,3
8,1,9,"(2, 4, 0, 4, 0, 5)",UP,"(1, 4, 0, 4, 0, 5)",-0.1,False,move,5,3
9,1,10,"(1, 4, 0, 4, 0, 5)",UP,"(0, 4, 1, 1, 0, 6)",10.0,False,food,6,4


,episode,step,state,action,next_state,reward,done,event,length,foods_eaten
71,1,72,"(0, 1, 0, 0, 2, 16)",LEFT,"(0, 0, 4, 3, 2, 17)",-2.0,True,timeout,17,15
122,2,51,"(5, 5, 0, 4, 3, 14)",UP,"(5, 5, 0, 4, 3, 14)",-10.0,True,self_collision,14,12
191,3,69,"(0, 0, 3, 2, 0, 17)",RIGHT,"(0, 0, 3, 2, 0, 17)",-10.0,True,self_collision,17,15


,episode,episode_reward,steps,foods_eaten,max_length,termination,max_steps
0,1,132.3,72,15,17,timeout,72
1,2,106.2,51,12,14,self_collision,72
2,3,134.7,69,15,17,self_collision,72


## 3. 로그 해석 포인트
- `event=food`가 나온 step에서는 길이가 늘고, 다음 state에서 `len` 값이 증가해야 한다.
- `event=wall_collision` 또는 `event=self_collision`은 환경의 실패 종료 규칙이 제대로 작동했다는 뜻이다.
- `event=timeout`은 벽에 부딪히지 않더라도 episode가 너무 길어졌기 때문에 강제로 종료된 경우다.

이처럼 heuristic agent 로그를 먼저 확인해 두면, 나중에 DQN이나 policy gradient를 붙일 때 문제가 환경 때문인지 학습기 때문인지 훨씬 구분하기 쉬워진다.
